# Decision Trees — Interpretable Models with Sharp Edges

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>QM47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/blob/main/notebooks/nb11_decision_trees.ipynb)

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain how a decision tree partitions feature space using axis-aligned splits, for **both classification and regression** targets.
2. Fit and visualize a `DecisionTreeClassifier` (Wisconsin Breast Cancer) and a `DecisionTreeRegressor` (California Housing) using `plot_tree`.
3. Diagnose **overfitting** in a tree by comparing training accuracy / R² against 5-fold cross-validation, and read the train-vs-CV gap as a regularization signal.
4. Compare a tuned tree against its linear analogue (Logistic Regression vs Ridge) on identical CV folds.
5. Choose the simplest competitive `max_depth` using the **one-standard-error rule** — the discipline you will reuse in nb12, nb13, and nb14's selection ceremony.

---

> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises** — Exercise 1 on the classification track and Exercise 2 on the regression track. Complete both before submitting your notebook.

---

## 💼 Why This Matters

This notebook runs **two parallel cases** end-to-end, because your final-project group's target may be either classification or regression. Watching the same algorithm work on both cuts your transfer cost in half.

### The classification case — State Health Department screening

Wisconsin Breast Cancer biopsy data, 569 patients, 30 cell-nucleus measurements per biopsy, target = malignant vs benign. The Department's existing screening tool flags about 10% of clinic visits for confirmatory testing; the screening logistic-regression model from nb06 lifts true-positive rate but the oncologists push back: *"I can't explain to a patient why the model flagged them. I need to see the decision logic."*

A decision tree gives them: **"if worst radius > 16.8 AND mean texture > 21.4, then malignant."** The path from root to leaf is an auditable flowchart any clinician can read aloud.

### The regression case — HomeValue Analytics property listings

California Housing data, 20,640 census tracts, 8 features (median income, house age, occupancy, location), target = median house value in USD 100K units. HomeValue Analytics wants to deploy a price-prediction model on its public-facing listings: *"What is this property worth, given comparable tracts in the area?"*

A regression tree gives the legal team something they can defend in court: **"this property's prediction is the average of 47 comparable tracts that match it on the seven splits below."** No coefficients to interpret, no extrapolation outside the training range — just an averaging rule with a documented provenance.

### The reference floor every tree-based model must clear

Week 2 closed with a disciplined comparison: tuned linear models versus their default counterparts under nb09's **CI-overlap test**. The verdict was unambiguous on both spines:

- **Classification:** `Pipeline([StandardScaler, LogisticRegression(C=1.0, max_iter=5000)])` — random-search over `C` produced no statistically clear winner (the tuned variant's CI overlaps the default's). The default ships.
- **Regression:** `Pipeline([StandardScaler, LinearRegression()])` (OLS) — a tuned Ridge `α` grid produced no statistically clear winner either. OLS ships, by simplicity.

Throughout nb11 → nb15 these two pipelines are the **Week-2 reference models** — the linear baselines that every tree, forest, and boosting variant has to beat by a CI-clear margin to justify its added complexity. They appear as `reference_clf` and `reference_reg` in the setup cell and are the comparison floor in every model-comparison plot from here through nb14's selection ceremony.

A question that often comes up here is *"why two cases instead of one?"* Half the cohort's M3 milestone is a classification project and half is regression. Walking both spines through nb11 → nb15 means the algorithms, diagnostics, and selection protocol all transfer to whichever case your group picked. The variable suffix `_clf` / `_reg` keeps the two namespaces visibly separate so a `RandomForestClassifier` never accidentally gets fit on housing data.

---

## 1. Setup — Imports, Datasets, and Plot Helpers

The setup cell loads both datasets, defines the suffix convention (`_clf`, `_reg`), and registers two small plot helpers (`plot_train_val_curve`, `plot_predicted_vs_actual`) that every section will call. Locking the random seed to `RANDOM_SEED = 474` (course number) means every fold split, every tree fit, and every plot is reproducible across machines.

> 💡 **Gemini Prompt:** "Set up imports for sklearn DecisionTreeClassifier, DecisionTreeRegressor, plot_tree, LogisticRegression, Ridge, cross_val_score, train_test_split, StratifiedKFold, KFold, load_breast_cancer, fetch_california_housing. Set RANDOM_SEED = 474 and define two helper functions: plot_train_val_curve(x_values, train, val_mean, val_std, xlabel, ylabel, title, ax) for overfitting diagnostics, and plot_predicted_vs_actual(y_true, y_pred, ax, title) for regression diagnostics."
>
> **After running, verify:**
> - [ ] `RANDOM_SEED = 474` printed back
> - [ ] Helpers `plot_train_val_curve` and `plot_predicted_vs_actual` defined and callable
> - [ ] No import errors
> - [ ] Display precision set to 4 digits


In [ ]:
# Setup — imports, seeds, display, plot helpers
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, fetch_california_housing, make_classification
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, mean_squared_error, r2_score
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.precision', 4)
plt.rcParams['figure.figsize'] = (10, 6)

RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)

# --- Course color convention (nb19 alignment) ---
CLF_COLOR = '#1f77b4'   # teal — classification accent
REG_COLOR = '#ff7f0e'   # orange — regression accent
GREY      = '#999999'

# --- Helper 1: train-vs-CV overfitting curve ---
def plot_train_val_curve(x_values, train, val_mean, val_std, xlabel, ylabel, title, ax,
                         color_train=GREY, color_val=CLF_COLOR):
    """Side-by-side train and CV-mean curves with CV std as error bars."""
    xs = list(range(len(x_values)))
    ax.plot(xs, train, marker='o', label='Train (full training set)',
            linewidth=2, color=color_train)
    ax.errorbar(xs, val_mean, yerr=val_std, marker='s',
                label='5-fold CV mean ± SD', linewidth=2, capsize=5, color=color_val)
    ax.set_xticks(xs)
    ax.set_xticklabels([str(v) for v in x_values])
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

# --- Helper 2: predicted-vs-actual scatter (the standard regression diagnostic) ---
def plot_predicted_vs_actual(y_true, y_pred, ax, title='Predicted vs Actual',
                             color=REG_COLOR):
    """Scatter of y_pred vs y_true with the y=x line of perfect prediction."""
    ax.scatter(y_true, y_pred, alpha=0.25, s=8, color=color)
    lo = float(min(np.min(y_true), np.min(y_pred)))
    hi = float(max(np.max(y_true), np.max(y_pred)))
    ax.plot([lo, hi], [lo, hi], 'k--', alpha=0.5, label='Perfect prediction')
    ax.set_xlabel('Actual')
    ax.set_ylabel('Predicted')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

# --- Week-2 reference models (the linear baselines that survived nb09's CI-overlap test) ---
# Classification:  default LogReg(C=1.0) — random-search over C produced no CI-clear winner.
# Regression:      OLS                   — tuned Ridge α grid produced no CI-clear winner.
reference_clf = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(C=1.0, random_state=RANDOM_SEED, max_iter=5000))
])
reference_reg = Pipeline([
    ('scaler', StandardScaler()),
    ('reg',    LinearRegression())
])

print(f"✓ RANDOM_SEED = {RANDOM_SEED}")
print(f"✓ Helpers defined: plot_train_val_curve, plot_predicted_vs_actual")
print(f"✓ Week-2 references defined: reference_clf (LogReg C=1.0), reference_reg (OLS)")
print(f"✓ Display precision: 4 digits")


**Reading the output:**

The setup cell does three jobs at once. First, it imports both `DecisionTreeClassifier` (for Wisconsin breast cancer) and `DecisionTreeRegressor` (for California housing) — the two algorithms we will pair-fit throughout the notebook. Second, it locks `RANDOM_SEED = 474` so every CV fold, every tree fit, and every plot reproduces identically across machines. Third, it defines two helper functions: `plot_train_val_curve` (the overfitting diagnostic you will see in Section 5) and `plot_predicted_vs_actual` (the standard regression-diagnostic scatter, mandatory for every regression model on the final poster).

**Key takeaway:** Defining helpers once at the top is the same data-communication discipline you will see in nb19 — write the rendering rule once, reuse it across every section, never duplicate matplotlib boilerplate cell to cell.

---

## 2. Decision Tree Intuition — Axis-Aligned Splits

A decision tree partitions the input space with a sequence of **axis-aligned binary splits**. At each node it asks a question of the form *"is feature `j` greater than threshold `t`?"* and routes the sample left or right. After a few splits the input space is carved into rectangles; every rectangle (a leaf) holds a constant prediction — the majority class for classification or the mean target value for regression.

The picture below uses a synthetic 2D classification dataset so you can *see* every split happen on a flat plane. The actual breast-cancer and housing data lives in 30 and 8 dimensions respectively, but the rule is identical — splits are axis-aligned, predictions are leaf-level constants, and the tree's depth controls how finely the space is partitioned.

In [ ]:
# Synthetic 2D dataset — visualize axis-aligned splits step by step.
X_toy, y_toy = make_classification(
    n_samples=200, n_features=2, n_redundant=0, n_informative=2,
    n_clusters_per_class=1, class_sep=1.0, random_state=RANDOM_SEED
)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, depth in zip(axes, [1, 2, 3, 6]):
    # Fit a tree at this depth
    t = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_SEED).fit(X_toy, y_toy)

    # Decision-region grid
    x_min, x_max = X_toy[:, 0].min() - 0.5, X_toy[:, 0].max() + 0.5
    y_min, y_max = X_toy[:, 1].min() - 0.5, X_toy[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    Z = t.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.25, cmap='RdBu')
    ax.scatter(X_toy[:, 0], X_toy[:, 1], c=y_toy, cmap='RdBu',
               edgecolor='k', s=40)
    ax.set_xlabel('feature 1')
    ax.set_ylabel('feature 2')
    ax.set_title(f'max_depth = {depth}  →  {t.get_n_leaves()} leaves',
                 fontsize=12, fontweight='bold')

fig.suptitle('Decision-tree partitions become finer as depth increases — every boundary is axis-aligned',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("💡 Every boundary is a horizontal or vertical line — never diagonal.")
print("💡 At depth 6 the tree is already memorizing individual points.")


**Reading the output:**

At `max_depth = 1` the tree makes exactly one split — a single horizontal or vertical line that divides the plane in two. At `max_depth = 2` it adds a second split inside one of the regions, producing a four-rectangle partition. By `max_depth = 6` the partitioning has become extremely fine — many of the leaves contain only one or two training points each, which is the visual signature of overfitting.

A question that often comes up here is *"why are tree boundaries always axis-aligned?"* Because each split asks a question about exactly one feature at a time (`feature_j > threshold`), the boundary it draws is perpendicular to that feature's axis. A linear model such as logistic regression draws a single diagonal line that uses both features at once. Trees can approximate a diagonal boundary with a staircase of axis-aligned splits, but the staircase costs depth — and depth costs variance, as Section 5 will make quantitative.

**Key takeaway:** Tree depth is the lever that controls bias and variance. Shallow trees underfit (large rectangles, smooth boundaries, high bias). Deep trees overfit (tiny rectangles, jagged boundaries, high variance). The rest of this notebook is about finding the right setting.

---

## 3. Load Both Datasets — Two Locked Test Sets

Both spines use the same locking discipline: split into train / test 70/30 with `random_state=RANDOM_SEED`, then **never touch the test set** for the rest of the notebook. All evaluation goes through 5-fold cross-validation on the training set — the same CV-first rule you will see enforced from here through nb18.

The classification track uses **stratified** CV (preserves the malignant/benign class balance per fold). The regression track uses plain `KFold` because there are no classes to balance.

> 💡 **Gemini Prompt:** "Load the breast cancer dataset, stratified 70/30 split with seed 474, into X_train_clf, X_test_clf, y_train_clf, y_test_clf, and create cv_clf = StratifiedKFold(5). Load California Housing, plain 70/30 split with seed 474, into X_train_reg, X_test_reg, y_train_reg, y_test_reg, and create cv_reg = KFold(5). Print sizes and remind that both test sets are LOCKED until nb14."
>
> **After running, verify:**
> - [ ] Classification: train ~398, test ~171 (out of 569)
> - [ ] Regression: train ~14,448, test ~6,192 (out of 20,640)
> - [ ] Both splits use random_state=RANDOM_SEED
> - [ ] Both splits print "LOCKED" in the output


In [ ]:
# --- Classification track: Wisconsin breast cancer (State Health Department) ---
data_clf = load_breast_cancer(as_frame=True)
X_clf = data_clf.data
y_clf = data_clf.target

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.3, random_state=RANDOM_SEED, stratify=y_clf
)
cv_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

# --- Regression track: California Housing (HomeValue Analytics) ---
data_reg = fetch_california_housing(as_frame=True)
X_reg = data_reg.data
y_reg = data_reg.target  # in units of USD 100K

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.3, random_state=RANDOM_SEED
)
cv_reg = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

print("=== CLASSIFICATION (State Health Department / Wisconsin Breast Cancer) ===")
print(f"  Train: {len(X_train_clf):>6} | Test: {len(X_test_clf):>6} (LOCKED until nb14)")
print(f"  Features: {X_clf.shape[1]}  | Class balance (train): "
      f"{y_train_clf.mean():.3f} positive (benign)")

print()
print("=== REGRESSION (HomeValue Analytics / California Housing) ===")
print(f"  Train: {len(X_train_reg):>6} | Test: {len(X_test_reg):>6} (LOCKED until nb14)")
print(f"  Features: {X_reg.shape[1]}  | Target: median house value, USD 100K units")
print(f"  y_train range: USD {y_train_reg.min()*100:.0f}K – USD {y_train_reg.max()*100:.0f}K, "
      f"mean USD {y_train_reg.mean()*100:.0f}K")


**Reading the output:**

Two parallel splits, two locked test sets, two CV splitters — one per case. Notice the size asymmetry: the classification training set has ~398 samples while the regression set has ~14,448 (about 36× larger). That difference matters for two reasons.

First, the same `max_depth` will overfit much harder on the small dataset than on the large one — 30 features and 398 samples means the tree can memorize individual patients with very few splits. Second, regression CV on 14k samples will be noticeably slower than classification CV on 400 samples; that is why the regression cells in this notebook will sometimes drop to 3-fold CV when 5-fold would not finish in a reasonable time on Colab.

A question that often comes up here is *"why is the test set 'locked' if I can see the variable in memory?"* The discipline is behavioral, not technical. Nothing stops you from typing `tree.predict(X_test_clf)` in a cell — but doing so forfeits the CV-first guarantee that lets nb14 pronounce a verdict on the locked test set with statistical meaning. Treat the test variables like a sealed envelope: they exist, but you do not open them until the ceremony in nb14.

**Key takeaway:** Two cases, two namespaces, two sealed envelopes. Everything from here through Section 7 evaluates on the **training** sets only.

---

## 4. Classification Tree Example — Wisconsin Breast Cancer

The first fit is the classification tree at `max_depth=3`. Three splits is enough to land a strong baseline on this dataset and shallow enough that `plot_tree` produces something a clinician can actually read on a single page. We will compare its training accuracy to its 5-fold CV accuracy and visualize the tree structure to confirm that the splits are clinically plausible.

> 💡 **Gemini Prompt:** "Fit a DecisionTreeClassifier(max_depth=3, random_state=474) on X_train_clf, y_train_clf. Print training accuracy on the full training set and 5-fold CV accuracy mean ± SD using cv_clf and scoring='accuracy'. Then call plot_tree with feature_names, class_names, filled=True."
>
> **After running, verify:**
> - [ ] Train accuracy in the 0.96–0.99 range
> - [ ] CV mean ± SD reported with 4-digit precision
> - [ ] Train-CV gap printed explicitly
> - [ ] Tree visualization shows ≤ 8 leaves


In [ ]:
# Classification tree at max_depth=3 — fit, score, visualize.
tree_clf = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_SEED)
tree_clf.fit(X_train_clf, y_train_clf)

train_acc_clf = tree_clf.score(X_train_clf, y_train_clf)
cv_scores_clf = cross_val_score(tree_clf, X_train_clf, y_train_clf,
                                cv=cv_clf, scoring='accuracy')
cv_mean_clf = cv_scores_clf.mean()
cv_std_clf  = cv_scores_clf.std(ddof=1)

print("=== CLASSIFICATION TREE (max_depth=3) ===")
print(f"Train accuracy:        {train_acc_clf:.4f}")
print(f"5-fold CV accuracy:    {cv_mean_clf:.4f}  (SD = {cv_std_clf:.4f})")
print(f"Overfit gap (train−CV): {train_acc_clf - cv_mean_clf:.4f}")
print(f"Tree leaves:           {tree_clf.get_n_leaves()}")

# --- Visualize the tree ---
fig, ax = plt.subplots(figsize=(20, 9))
plot_tree(
    tree_clf,
    feature_names=X_clf.columns,
    class_names=data_clf.target_names,
    filled=True,
    rounded=True,
    fontsize=10,
    ax=ax,
)
ax.set_title('Wisconsin Breast Cancer — Classification Tree at max_depth=3',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Each leaf shows: gini impurity, sample count, class distribution, predicted class.")
print("💡 Darker shading = purer leaf (mostly one class).")


**Reading the output:**

A `max_depth=3` tree on this dataset typically lands a 5-fold CV accuracy around 0.93–0.95 with an overfit gap of 0.02–0.04 — a healthy sign. The tree itself uses only seven or eight leaves; you can read every decision path in plain English. The root split is almost always on `worst radius` or `worst perimeter`, both of which are well-established malignancy markers in the clinical literature, which is the kind of plausibility check that builds trust with the State Health Department's review board.

A question that often comes up here is *"what does the gini number on each node mean?"* Gini impurity is `1 − Σ p_i²` where `p_i` is the fraction of samples in class `i`. A pure leaf (all malignant or all benign) has gini = 0. A perfectly mixed leaf (50/50) has gini = 0.5. The tree's split criterion at every internal node is "pick the feature and threshold that drops the **weighted average gini** of the two children most." That single objective — minimize child impurity — is the entirety of how a classification tree decides where to cut.

**Key takeaway:** A shallow classification tree is interpretable, plausible, and already competitive — but the CV gap of 2–4 points tells you the tree could go deeper. Section 5 will tune that depth properly.

---

## 5. Regression Tree Example — California Housing

The regression tree on California Housing has the same machinery — recursive binary splits, leaf-level constant predictions — with two changes:

1. The **split criterion** is mean squared error inside each child, not gini impurity. The tree picks the feature and threshold that drops weighted child-MSE the most.
2. The **leaf prediction** is the average target value of the training samples that fall into that leaf, not a class vote.

The visualization below shows three things in parallel: the fitted tree (top), a predicted-vs-actual scatter (bottom left), and a step-function plot showing how the tree predicts as a function of `MedInc` alone (bottom right). The step plot makes the "axis-aligned partitioning" idea visceral for a 1D regression problem.

> 💡 **Gemini Prompt:** "Fit DecisionTreeRegressor(max_depth=3, random_state=474) on X_train_reg, y_train_reg. Print train R², 5-fold CV R² mean ± SD using cv_reg, and train RMSE in dollars. Plot three panels: plot_tree, predicted-vs-actual scatter (use the helper), and a step-function showing tree.predict over MedInc (with other features held at their training median). Use cross_val_score with scoring='neg_root_mean_squared_error' for the RMSE CV."
>
> **After running, verify:**
> - [ ] Train R² in 0.50–0.70 range; CV R² slightly lower
> - [ ] CV RMSE reported in USD (multiply by 100,000)
> - [ ] Step-function plot shows 8 distinct horizontal levels (one per leaf)
> - [ ] Predicted-vs-actual scatter has the y=x reference line drawn


In [ ]:
# Regression tree at max_depth=3 — fit, score, visualize, step-function.
tree_reg = DecisionTreeRegressor(max_depth=3, random_state=RANDOM_SEED)
tree_reg.fit(X_train_reg, y_train_reg)

train_r2_reg  = tree_reg.score(X_train_reg, y_train_reg)
cv_r2_reg     = cross_val_score(tree_reg, X_train_reg, y_train_reg,
                                cv=cv_reg, scoring='r2')
cv_rmse_reg   = -cross_val_score(tree_reg, X_train_reg, y_train_reg,
                                 cv=cv_reg, scoring='neg_root_mean_squared_error')

print("=== REGRESSION TREE (max_depth=3) ===")
print(f"Train R²:               {train_r2_reg:.4f}")
print(f"5-fold CV R²:           {cv_r2_reg.mean():.4f}  (SD = {cv_r2_reg.std(ddof=1):.4f})")
print(f"5-fold CV RMSE:         {cv_rmse_reg.mean():.4f} (in 100K USD units)")
print(f"5-fold CV RMSE in USD:  USD {cv_rmse_reg.mean()*100_000:,.0f}")
print(f"Tree leaves:            {tree_reg.get_n_leaves()}")

# --- Three-panel diagnostic ---
fig = plt.figure(figsize=(20, 11))
gs  = fig.add_gridspec(2, 2, height_ratios=[1.4, 1])

# Top: the regression tree itself
ax_tree = fig.add_subplot(gs[0, :])
plot_tree(
    tree_reg,
    feature_names=X_reg.columns,
    filled=True, rounded=True, fontsize=10, ax=ax_tree,
)
ax_tree.set_title('California Housing — Regression Tree at max_depth=3',
                  fontsize=14, fontweight='bold')

# Bottom-left: predicted-vs-actual on training set
ax_pva = fig.add_subplot(gs[1, 0])
y_pred_reg = tree_reg.predict(X_train_reg)
plot_predicted_vs_actual(y_train_reg, y_pred_reg, ax_pva,
                         title='Predicted vs Actual (training set, USD 100K units)')

# Bottom-right: 1D step function — predict as MedInc varies, others at median
ax_step = fig.add_subplot(gs[1, 1])
medinc_range = np.linspace(X_train_reg['MedInc'].min(),
                           X_train_reg['MedInc'].max(), 500)
median_row = X_train_reg.median()
grid = pd.DataFrame([median_row] * 500, columns=X_train_reg.columns)
grid['MedInc'] = medinc_range
y_step = tree_reg.predict(grid)

ax_step.scatter(X_train_reg['MedInc'], y_train_reg,
                alpha=0.10, s=6, color=GREY, label='Training data')
ax_step.plot(medinc_range, y_step, color=REG_COLOR, linewidth=2.5,
             label='Tree prediction (other features at median)')
ax_step.set_xlabel('MedInc (median income, 10K USD units)')
ax_step.set_ylabel('Predicted house value (USD 100K units)')
ax_step.set_title('Tree predicts step-functions, not smooth curves',
                  fontsize=12, fontweight='bold')
ax_step.legend()
ax_step.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 The step function has exactly as many levels as the tree has leaves.")
print("💡 This is the visual signature of axis-aligned splits — no smoothness.")


**Reading the output:**

A `max_depth=3` regression tree on California Housing usually lands a CV R² around 0.50 and a CV RMSE around USD 80K. That is far from state-of-the-art on this benchmark — Random Forests and Gradient Boosting (next two notebooks) will roughly halve the RMSE — but as a baseline it already beats the *"predict the training mean"* benchmark by a wide margin, which is what HomeValue's pricing analysts had been doing manually before this engagement.

The step-function plot in the bottom-right panel is the visual signature of every regression tree. Across the entire range of `MedInc`, the prediction takes only as many distinct values as the tree has leaves — eight, in this case. Within each step the tree is constant; at the boundaries it jumps. This means a regression tree **cannot extrapolate**. If a new property has a `MedInc` higher than anything seen in training, the tree clamps to the highest-leaf average; it has no notion of "the price keeps rising past USD 500K." Linear models can extrapolate along a regression line; trees cannot. This trade-off — interpretability and non-linearity in exchange for boundedness and step-function predictions — is the central characteristic of the algorithm.

A question that often comes up here is *"if the regression tree only predicts step values, isn't that always worse than a linear model?"* Not always. When the true relationship is **non-monotonic** (price drops in some MedInc ranges due to neighborhood effects), the tree captures that bend; the linear model cannot. When the relationship is **monotonic and smooth**, the linear model wins. Section 7 (Tree vs Linear Model) makes that comparison concrete on both spines.

**Key takeaway:** Regression trees give you interpretability and non-linearity at the cost of step-function predictions and zero extrapolation. Whether that trade is worth it depends on the data — which is what the next two sections quantify.

---

## 6. The Overfitting Problem — Paired Across Both Cases

The same depth-vs-performance pattern holds for both classification and regression: shallow trees underfit, deep trees memorize, and somewhere in the middle is the sweet spot. The two-panel plot below sweeps `max_depth ∈ {1, 2, 3, 5, 10, 20, None}` and reports training score versus 5-fold CV score for each case side-by-side.

The headline reading is the **train–CV gap**: when the gap explodes (train near 1.0, CV plateaus or drops), the tree has memorized noise that does not transfer.

> 💡 **Gemini Prompt:** "Sweep max_depth in [1, 2, 3, 5, 10, 20, None] for both DecisionTreeClassifier on X_train_clf and DecisionTreeRegressor on X_train_reg. For each depth and each case, fit on full training set, record train score and 5-fold CV mean ± SD. Build a 1×2 subplot using the plot_train_val_curve helper — left panel classification (accuracy), right panel regression (R²). Title the figure to make the universality of the overfitting pattern explicit."
>
> **After running, verify:**
> - [ ] Both panels share the same x-axis layout (same depth values)
> - [ ] Classification curve plateaus near 0.95+ around depth 3–5
> - [ ] Regression curve plateaus around depth 8–10
> - [ ] Train curves both reach 1.0 at unrestricted depth (None)


In [ ]:
# Paired depth sweep — same pattern emerges on both cases.
depths = [1, 2, 3, 5, 10, 20, None]

# --- Classification sweep ---
clf_train, clf_val_mean, clf_val_std = [], [], []
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_SEED).fit(X_train_clf, y_train_clf)
    clf_train.append(t.score(X_train_clf, y_train_clf))
    s = cross_val_score(t, X_train_clf, y_train_clf, cv=cv_clf, scoring='accuracy')
    clf_val_mean.append(s.mean())
    clf_val_std.append(s.std(ddof=1))

# --- Regression sweep ---
reg_train, reg_val_mean, reg_val_std = [], [], []
for d in depths:
    t = DecisionTreeRegressor(max_depth=d, random_state=RANDOM_SEED).fit(X_train_reg, y_train_reg)
    reg_train.append(t.score(X_train_reg, y_train_reg))
    s = cross_val_score(t, X_train_reg, y_train_reg, cv=cv_reg, scoring='r2')
    reg_val_mean.append(s.mean())
    reg_val_std.append(s.std(ddof=1))

# --- Side-by-side panels ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plot_train_val_curve(
    [str(d) for d in depths], clf_train, clf_val_mean, clf_val_std,
    xlabel='max_depth', ylabel='Accuracy',
    title='Classification — Wisconsin Breast Cancer',
    ax=axes[0], color_val=CLF_COLOR
)
plot_train_val_curve(
    [str(d) for d in depths], reg_train, reg_val_mean, reg_val_std,
    xlabel='max_depth', ylabel='R²',
    title='Regression — California Housing',
    ax=axes[1], color_val=REG_COLOR
)

fig.suptitle('Same overfitting pattern on both spines — train rises to 1.0; CV plateaus and then degrades',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Print the train–CV gap table
print("=== TRAIN − CV GAP BY DEPTH ===")
print(f"{'depth':>8} | {'clf train':>10} | {'clf CV':>10} | {'gap':>8} ||"
      f" {'reg train':>10} | {'reg CV':>10} | {'gap':>8}")
for i, d in enumerate(depths):
    print(f"{str(d):>8} | {clf_train[i]:>10.4f} | {clf_val_mean[i]:>10.4f} | "
          f"{clf_train[i]-clf_val_mean[i]:>8.4f} || "
          f"{reg_train[i]:>10.4f} | {reg_val_mean[i]:>10.4f} | "
          f"{reg_train[i]-reg_val_mean[i]:>8.4f}")


**Reading the output:**

The two panels tell the same story at different scales. On the left, classification accuracy rises steeply from depth 1 to depth 3, then plateaus near 0.93. On the right, regression R² rises from 0.30 at depth 1 to a plateau near 0.65 at depth 8–10. In **both** cases, the training score keeps climbing toward 1.0 even after the CV score has stopped improving — that gap is the pure-overfitting signature.

Notice the asymmetry in *where* the plateau lands. Classification levels off around depth 3–5 because there are only 30 features and ~400 training samples — there is not much room for the tree to grow before it starts memorizing. Regression plateaus around depth 8–10 because there are 14k training samples — the tree has more data to generalize from before memorization kicks in. The plateau depth is roughly the depth at which the leaves stop containing enough samples to be statistically reliable, and that depends on dataset size, not on the algorithm.

A question that often comes up here is *"why does CV degrade after the plateau in some sweeps and stay flat in others?"* When CV degrades after a plateau, it is because the deep tree has carved out tiny leaves that fit noise specific to the training set; on the held-out folds those leaves predict the wrong class or the wrong leaf-mean. When CV stays flat after a plateau, the tree is overfitting too — but the overfitting is symmetric across folds and so the average CV score does not move much. Either way the **train–CV gap** is the more reliable diagnostic: it grows monotonically with depth on every dataset.

**Key takeaway:** The overfitting pattern is universal — same shape, different scales. Train score keeps rising, CV plateaus, gap explodes. The right depth is wherever CV is highest and the gap is acceptably small. Section 7 makes this comparison against the linear analogue; Exercises 1 and 2 will ask you to pick the optimal depth on each spine using the one-standard-error rule.

---

## 7. Tree vs Week-2 Reference — Paired

Now compare each tree to the **Week-2 reference model** for its track — the linear baseline that survived nb09's CI-overlap discipline. These are not arbitrary linear analogues; they are the exact pipelines your group's M2 baseline (or its sibling) shipped at the end of last week:

- **Classification:** `DecisionTreeClassifier(max_depth=3)` vs `reference_clf` = `Pipeline([StandardScaler, LogisticRegression(C=1.0, max_iter=5000)])`. Primary metric: ROC-AUC.
- **Regression:** `DecisionTreeRegressor(max_depth=5)` vs `reference_reg` = `Pipeline([StandardScaler, LinearRegression()])` (OLS). Primary metric: R².

The headline is two side-by-side bar charts with CV mean and SD. The same CI-overlap rule from nb08 / nb09 applies — if the tree's SD interval overlaps the reference's, the tree has not earned the right to displace the simpler linear baseline.

> 💡 **Gemini Prompt:** "Compare DecisionTreeClassifier(max_depth=3) against reference_clf (the Week-2 LogReg pipeline) on the breast cancer training set using cv_clf and scoring='roc_auc'. Compare DecisionTreeRegressor(max_depth=5) against reference_reg (the Week-2 OLS pipeline) on the California housing training set using cv_reg and scoring='r2'. Two side-by-side bar charts with CV mean ± SD error bars; label the reference bars 'Week-2 reference' and the tree bars 'Decision Tree'."
>
> **After running, verify:**
> - [ ] Classification ROC-AUC mean reported for both Tree and reference_clf
> - [ ] Regression R² mean reported for both Tree and reference_reg
> - [ ] Error bars use CV standard deviation (ddof=1)
> - [ ] Reference bars carry the label "Week-2 reference: LogReg(C=1.0)" / "Week-2 reference: OLS"


In [ ]:
# Tree vs Week-2 reference — paired comparison.
clf_models = {
    'Decision Tree (depth=3)':              DecisionTreeClassifier(max_depth=3, random_state=RANDOM_SEED),
    'Week-2 reference: LogReg(C=1.0)':      reference_clf,
}
reg_models = {
    'Decision Tree (depth=5)':              DecisionTreeRegressor(max_depth=5, random_state=RANDOM_SEED),
    'Week-2 reference: OLS':                reference_reg,
}

clf_results = []
for name, m in clf_models.items():
    s = cross_val_score(m, X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc')
    clf_results.append({'model': name, 'mean': s.mean(), 'sd': s.std(ddof=1)})

reg_results = []
for name, m in reg_models.items():
    s = cross_val_score(m, X_train_reg, y_train_reg, cv=cv_reg, scoring='r2')
    reg_results.append({'model': name, 'mean': s.mean(), 'sd': s.std(ddof=1)})

clf_df = pd.DataFrame(clf_results)
reg_df = pd.DataFrame(reg_results)

print("=== CLASSIFICATION (5-fold CV ROC-AUC) ===")
print(clf_df.to_string(index=False))
print()
print("=== REGRESSION (5-fold CV R²) ===")
print(reg_df.to_string(index=False))

# Side-by-side bar plots
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

ax = axes[0]
ax.bar(clf_df['model'], clf_df['mean'], yerr=clf_df['sd'], capsize=10,
       color=[CLF_COLOR, GREY], edgecolor='black')
for i, row in clf_df.iterrows():
    ax.text(i, row['mean'] + 0.005, f"{row['mean']:.4f}", ha='center', fontsize=10)
ax.set_ylabel('5-fold CV ROC-AUC')
ax.set_title('Classification — Tree vs Week-2 reference', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

ax = axes[1]
ax.bar(reg_df['model'], reg_df['mean'], yerr=reg_df['sd'], capsize=10,
       color=[REG_COLOR, GREY], edgecolor='black')
for i, row in reg_df.iterrows():
    ax.text(i, row['mean'] + 0.005, f"{row['mean']:.4f}", ha='center', fontsize=10)
ax.set_ylabel('5-fold CV R²')
ax.set_title('Regression — Tree vs Week-2 reference', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

fig.suptitle('Tree vs Week-2 reference — paired CV comparison',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Reading the output:**

On the breast-cancer dataset, the classification tree at depth 3 typically reaches CV ROC-AUC around 0.96 and the Week-2 reference (LogReg C=1.0) lands around 0.99. The tree's error bar overlaps the reference's — by the CI-overlap rule from nb08 / nb09 the two are within statistical-tie territory, but the reference has a small, consistent edge. This is what you would expect on a dataset that is approximately linearly separable in the right feature space, which the cell-nucleus measurements largely are after standardization. **The single tree has not earned the right to displace the Week-2 reference yet** — it is competitive, but not convincingly better.

On the California-housing dataset, the regression tree at depth 5 typically reaches CV R² around 0.62 and the Week-2 reference (OLS) lands around 0.60. Here the tree wins by mean, but only by ~2 R² points, and the SD across folds is wide enough that the two are statistically close. The reason is that California housing has moderate non-linearity (price-vs-income curves, location effects) that the tree can capture and OLS cannot — but the dataset is large enough and the noise floor high enough that the linear baseline holds its own. **The single regression tree's lift over the Week-2 reference is not yet CI-clear** — it is suggestive, not decisive.

A question that often comes up here is *"if the Week-2 reference is so close, why teach trees at all?"* Three answers. First, today's single tree is the floor for the next two notebooks: Random Forests (nb12) and Gradient Boosting (nb13) are ensembles **of trees**, not ensembles of linear models, and they routinely lift CV scores well past the linear baseline. The reference floor will not move; the trees will rise above it. Second, trees give you **interpretability** that the linear baseline does not — the State Health Department's clinicians can trace a path through the tree but cannot read 30 logistic coefficients in their heads. Third, the comparison itself is the discipline: knowing how to benchmark every new model against a CV-stable reference is exactly what lets nb14's selection ceremony declare a defensible champion across five candidates per spine.

**Key takeaway:** A single tree is usually competitive with the Week-2 reference, rarely convincingly better, and almost always more interpretable in shape. Its primary value going forward is as the building block for ensembles — and as the visible reminder that **every new model has to clear the Week-2 reference floor by a CI-clear margin** before it earns a place on the candidate roster.

---

## 📝 PAUSE-AND-DO Exercise 1 (clf, 5 minutes) — Tune the Classification Tree's Depth

**Task:** Use 5-fold cross-validation to pick the optimal `max_depth` for `DecisionTreeClassifier` on the Wisconsin breast cancer training set, scoring on **ROC-AUC**.

**Instructions:**
1. Sweep `max_depth ∈ [2, 3, 4, 5, 6, 7, 8, 10, 15]`.
2. For each depth, compute the 5-fold CV mean and SD using `cv_clf`.
3. Plot CV mean with SD error bars across depths.
4. Report the depth with the highest CV mean **and** the simplest depth whose CV mean is within one SD of the best (the **one-standard-error rule**).
5. Write 3 short findings: what does the curve tell you about the bias-variance trade-off?

---

> 💡 **Gemini Prompt:** "Cross-validate DecisionTreeClassifier with random_state=474 over depths [2,3,4,5,6,7,8,10,15] on X_train_clf, y_train_clf using cv_clf and scoring='roc_auc'. Report mean ± SD for each depth in a DataFrame, plot mean with SD error bars, mark the highest-mean depth with a dashed line, and apply the one-standard-error rule to pick the simplest competitive depth."
>
> **After running, verify:**
> - [ ] DataFrame has 9 rows (one per depth)
> - [ ] Plot uses error bars from the SD column
> - [ ] Best depth and one-SE-rule depth both reported
> - [ ] CV ROC-AUC values are in 0.92–0.99 range


In [ ]:
# YOUR SOLUTION CODE HERE
# Tune classification-tree depth using 5-fold CV ROC-AUC.
# Apply the one-standard-error rule when selecting the final depth.


## 📝 PAUSE-AND-DO Exercise 2 (reg, 5 minutes) — Tune the Regression Tree's Depth

**Task:** Use 5-fold cross-validation to pick the optimal `max_depth` for `DecisionTreeRegressor` on the California housing training set, scoring on **R²**.

**Instructions:**
1. Sweep `max_depth ∈ [2, 3, 5, 7, 10, 15, 20]`.
2. For each depth, compute the 5-fold CV mean and SD using `cv_reg`.
3. Plot CV mean with SD error bars across depths.
4. Apply the same one-standard-error rule to pick the simplest competitive depth.
5. Write 3 short findings comparing this regression sweep to the classification sweep in Exercise 1 — same shape, different scale?

---

> 💡 **Gemini Prompt:** "Cross-validate DecisionTreeRegressor with random_state=474 over depths [2,3,5,7,10,15,20] on X_train_reg, y_train_reg using cv_reg and scoring='r2'. Report mean ± SD for each depth in a DataFrame, plot mean with SD error bars, mark the highest-mean depth with a dashed line, and apply the one-standard-error rule. Convert the best CV-RMSE to USD (multiply by 100,000) and print it."
>
> **After running, verify:**
> - [ ] DataFrame has 7 rows (one per depth)
> - [ ] Plot uses error bars from the SD column
> - [ ] Best depth and one-SE-rule depth both reported
> - [ ] CV R² values are in 0.50–0.75 range
> - [ ] Best CV RMSE printed in USD


In [ ]:
# YOUR SOLUTION CODE HERE
# Tune regression-tree depth using 5-fold CV R².
# Apply the one-standard-error rule and report best CV-RMSE in USD.


## 8. When to Use Decision Trees

Both spines now have a clear picture of where a single tree wins and where it loses. The summary below crystallizes the decision criteria you will reuse in nb12 (forests build on trees), nb13 (boosting builds on trees), and nb14 (the selection ceremony has both tree-based and linear candidates).

In [ ]:
# Visual summary: when does a tree win?
criteria = [
    ('Linearly separable data',                          False, True),
    ('Strong feature interactions',                       True,  False),
    ('Need for interpretable decision logic',             True,  False),
    ('Need to extrapolate beyond training range',         False, True),
    ('Mixed feature types (numeric + categorical)',       True,  False),
    ('Very small training set (<200 samples)',            False, True),
    ('Smooth target relationship',                        False, True),
    ('Non-linear, non-monotonic relationships',           True,  False),
]

df_criteria = pd.DataFrame(criteria, columns=['Criterion', 'Tree wins', 'Linear wins'])
print("=== When does a single decision tree win against its linear analogue? ===")
print(df_criteria.to_string(index=False))

# Plot as a 2-column heatmap
fig, ax = plt.subplots(figsize=(10, 4.5))
mat = df_criteria[['Tree wins', 'Linear wins']].astype(int).values
ax.imshow(mat, cmap='RdYlGn', aspect='auto', alpha=0.7)
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        ax.text(j, i, '✓' if mat[i, j] else '', ha='center', va='center',
                fontsize=18, color='black')
ax.set_xticks([0, 1])
ax.set_xticklabels(['Tree wins', 'Linear wins'], fontsize=11)
ax.set_yticks(range(len(df_criteria)))
ax.set_yticklabels(df_criteria['Criterion'], fontsize=10)
ax.set_title('When to reach for a tree vs a linear model — the summary card',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


**Reading the output:**

The card collapses Section 7's empirical comparison into a checklist. Trees beat linear models on **interaction-heavy, mixed-type, non-monotonic, interpretability-critical** problems. Linear models beat trees on **smooth, monotonic, extrapolating, small-data** problems. Neither wins universally — which is precisely why nb14's selection ceremony evaluates both families on identical CV folds before declaring a champion.

**Key takeaway:** A single tree is rarely the final answer; it is the building block for the ensembles that come next. Forests average many trees to reduce variance (nb12). Gradient boosting trains trees sequentially to reduce bias (nb13). Both inherit the interpretability story from this notebook; both lift the performance ceiling far beyond what a single tree can reach.

---

## 9. Wrap-Up — Key Takeaways

**What landed today:**

1. **Decision trees partition feature space with axis-aligned splits.** Every leaf holds a constant prediction — majority class for classification, leaf-mean for regression. Tree depth controls how finely the space is partitioned.
2. **The overfitting pattern is universal.** On both spines, training score climbs toward 1.0 while CV score plateaus. The train–CV gap is the regularization signal; pick the simplest depth whose CV mean is within one SD of the best (one-standard-error rule).
3. **Single trees are interpretable but rarely best.** They beat linear models on interaction-heavy, mixed-type, non-monotonic data; they lose on smooth, linear, extrapolating data. The CV-bar comparison from Section 7 is the reusable diagnostic for that decision.
4. **Two parallel spines, two namespaces, two locked test sets.** From here through nb14, every section of every notebook will pair the classification and regression tracks under the `_clf` / `_reg` suffix convention. The discipline keeps the two cases unconfusable while you work.

**Bridge to nb12 — Random Forests:**

A single tree is high-variance: a small change in the training data can flip the root split. Random forests fix that by **averaging many trees**, each trained on a bootstrap sample with a random feature subset. The same dual-case structure continues — `RandomForestClassifier` on Wisconsin breast cancer, `RandomForestRegressor` on California housing, with paired diagnostics at every step. nb12 also introduces the **four-method feature-importance reconciliation table** that nb15 will lean on for interpretation. Bring today's depth-selection muscle memory with you — forests need to pick depth, n_estimators, and max_features under the same one-SE rule.

A question that often comes up at this point is *"if forests beat trees on every benchmark, why teach single trees first?"* Two reasons. First, the forest is an average of trees — without understanding what one tree does, the averaging operation is opaque. Second, single trees are still the right answer when the deliverable demands a single auditable decision path (clinical screening, regulatory compliance, courtroom defense). Forests trade interpretability for stability; that trade is sometimes worth making and sometimes not, and you cannot evaluate the trade without first having seen what gets lost.

---

## Participation Assignment Submission Instructions

### To Submit This Notebook:

1. **Complete both PAUSE-AND-DO exercises** — Exercise 1 (classification depth tuning) and Exercise 2 (regression depth tuning).
2. **Run All Cells** — `Runtime → Run all` to ensure every cell executes without error.
3. **Save a Copy** — `File → Save a copy in Drive`, or download as `.ipynb`.
4. **Submit** — upload the `.ipynb` file to the Notebook 11 participation assignment on Brightspace.

### Before Submitting, Check:

- [ ] Both exercise solutions produce a CV-CI plot with the chosen depth marked
- [ ] All figures render (none broken)
- [ ] Both `_clf` and `_reg` variable namespaces stay disjoint (no `NameError`)
- [ ] You can defend the one-SE-rule pick on each spine in plain English

### Next Step:

- **Notebook 12** — Random Forests + Importance (Day 12)

---

<center>

**Thank you!**

</center>